In [1]:
# Importing pandas for data manipulation
import pandas as pd

# Importing matplotlib for basic charts
import matplotlib.pyplot as plt

# Importing seaborn for beautiful charts
import seaborn as sns

# Importing os for folder creation
import os

# Setting seaborn style for all charts
sns.set_theme(style='whitegrid')

print("Libraries imported successfully ✅")

Libraries imported successfully ✅


In [ ]:
# Loading the CO2 dataset
co2 = pd.read_csv('data/data.csv')

# Loading the Nepal disaster dataset
disasters = pd.read_excel('data/nepal_disasters.xlsx')

# Quick look at CO2 data
print("CO2 Dataset Shape:", co2.shape)
print(co2.head(3))

# Quick look at disaster data
print("\nDisaster Dataset Shape:", disasters.shape)
print(disasters.head(3))

In [ ]:
# Checking disaster types in Nepal
print("Disaster Types:")
print(disasters['Disaster Type'].value_counts())

print("\nKey columns preview:")
print(disasters[['Start Year', 'Disaster Type', 'Disaster Subtype', 
                  'Total Deaths', 'No. Affected', 
                  'Total Damage (\'000 US$)']].head(10))

In [ ]:
# Filtering only flood related disasters
floods = disasters[disasters['Disaster Type'] == 'Flood'].copy()

# Also include landslides — caused by same climate patterns
landslides = disasters[disasters['Disaster Type'] == 'Mass movement (wet)'].copy()

# Combining floods and landslides
nepal_disasters = pd.concat([floods, landslides])

print("Flood records:", len(floods))
print("Landslide records:", len(landslides))
print("Total records:", len(nepal_disasters))

# Cleaning CO2 data — Nepal only
nepal_co2 = co2[co2['country'] == 'Nepal'][['year', 'co2', 'co2_per_capita']].dropna()
print("\nNepal CO2 records:", len(nepal_co2))

# Cleaning CO2 data — Global
co2_clean = co2[co2['year'] >= 2000]
global_avg = co2_clean.groupby('year')['co2_per_capita'].mean().reset_index()
print("Global average records:", len(global_avg))

In [ ]:
# Creating visuals folder
os.makedirs('visuals', exist_ok=True)

# CHART 1 — Nepal CO2 vs Global Average Per Capita
fig, ax = plt.subplots(figsize=(10, 6))

# Global average line
ax.plot(global_avg['year'], global_avg['co2_per_capita'],
        color='crimson', linewidth=2.5, label='Global Average')

# Nepal line
nepal_co2_filtered = nepal_co2[nepal_co2['year'] >= 2000]
ax.plot(nepal_co2_filtered['year'], nepal_co2_filtered['co2_per_capita'],
        color='green', linewidth=2.5, label='Nepal')

# Fill the gap
ax.fill_between(nepal_co2_filtered['year'],
                nepal_co2_filtered['co2_per_capita'],
                global_avg['co2_per_capita'],
                alpha=0.2, color='orange',
                label='Injustice Gap')

ax.set_title("Nepal Emits Almost Nothing\nBut Faces the Worst Climate Impacts",
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('CO2 Per Capita (Tonnes/Person)', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('visuals/nepal_vs_world_co2.png', dpi=150)
plt.show()

In [ ]:
# CHART 2 — Nepal Flood Deaths by Year
flood_deaths = floods.groupby('Start Year')['Total Deaths'].sum().reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

sns.barplot(data=flood_deaths, x='Start Year', 
            y='Total Deaths', color='crimson', ax=ax)

ax.set_title('Nepal Flood Deaths by Year (2000-2025)\nThe Human Cost of Climate Change',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Total Deaths', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('visuals/flood_deaths.png', dpi=150)
plt.show()

In [ ]:
# CHART 3 — People Affected by Floods
flood_affected = floods.groupby('Start Year')['No. Affected'].sum().reset_index()
flood_affected = flood_affected.dropna()

fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(flood_affected['Start Year'],
                flood_affected['No. Affected'],
                alpha=0.6, color='steelblue')
ax.plot(flood_affected['Start Year'],
        flood_affected['No. Affected'],
        color='navy', linewidth=2)

ax.set_title('People Affected by Floods in Nepal (2000-2025)\nMillions Displaced Every Year',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Number of People Affected', fontsize=12)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('visuals/flood_affected.png', dpi=150)
plt.show()

In [ ]:
# Stronger filter + Nepal visibility fix
co2_countries2 = co2_clean2[~co2_clean2['country'].str.contains(
    'excl|GCP|income|OECD|European Union|Middle East|World|'
    'Asia|Europe|Africa|Oceania|America|Antarctica|transport',
    case=False
)]

# Top 10 real countries
top10 = co2_countries2.groupby('country')['co2'].sum()
top10 = top10.sort_values(ascending=False).head(10)
top10 = top10.drop('International shipping', errors='ignore')

# Nepal total
nepal_total = co2_countries2[co2_countries2['country'] == 'Nepal']['co2'].sum()

# Combine
comparison = top10.copy()
comparison['Nepal 🇳🇵'] = nepal_total
comparison = comparison.sort_values(ascending=True)

# Colors
colors = ['green' if x == 'Nepal 🇳🇵' else 'crimson'
          for x in comparison.index]

fig, ax = plt.subplots(figsize=(10, 8))
comparison.plot(kind='barh', color=colors, ax=ax)

# Add Nepal value label
nepal_val = comparison['Nepal 🇳🇵']
ax.text(nepal_val + 500, comparison.index.get_loc('Nepal 🇳🇵'),
        f'{nepal_val:.0f} MT', va='center', color='green', fontweight='bold')

ax.set_title('Top Emitters vs Nepal\nWho is Responsible for Climate Change?',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total CO2 Emissions (Million Tonnes)', fontsize=12)
plt.tight_layout()
plt.savefig('visuals/top_emitters_vs_nepal.png', dpi=150)
plt.show()

In [ ]:
# CHART 5 — All Disaster Types in Nepal
disaster_summary = disasters.groupby('Disaster Type')['Total Deaths'].sum()
disaster_summary = disaster_summary.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['crimson' if x == 'Flood' else 'steelblue' 
          for x in disaster_summary.index]

sns.barplot(x=disaster_summary.values, 
            y=disaster_summary.index,
            palette=colors, ax=ax)

ax.set_title('Deaths by Disaster Type in Nepal (2000-2024)\nFloods Are the Deadliest Threat',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Total Deaths', fontsize=12)
ax.set_ylabel('Disaster Type', fontsize=12)
plt.tight_layout()
plt.savefig('visuals/disaster_types.png', dpi=150)
plt.show()